# Deliverable 2 - INF201 

Submission deadline: 30th Oct 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit the Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebooks
    * Should contain code, results/plots and discussion in markdown cells
* Also submit the output files generated by your code
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together

## Scenario  - Campus Coffee Shop Sales Pipeline
You have been hired as a Python developer for the Campus Coffee Shop Chain. The chain operates several coffee stands across different campus buildings. This is an international campus, so the stands accept payment in several currencies.

Each cash register saves transaction logs as text files. However, the legacy system has several issues:
* Mixed Character Encodings: Older registers write text files using ISO-8859-1 (Latin-1) encoding, while newer registers write files using UTF-8.
* System Noise: Registers intersperse valid sales lines with printer and network debug log lines.
* Nested Folder Hierarchy: Log files are stored across nested subdirectories organized by year, store, and register ID.
* Data Errors: System glitches record occasional extreme price outliers (e.g., $999.00), and prices in NOK are by error registered using EUR-scale values (e.g., 4.50 NOK instead of 45.00 NOK).

Your task is to set up Python code that discovers all log files, cleans the transaction data, calculates price metrics and generates summary reports.<br>
Note: In this assignment, you write the functions you will use in tasks 1-4, and then put everything together in task 5. You should therefore read through the entire assignment text before you start, to fully understand what you will use each function for.

### Overview of inputs and outputs for this assignment
Inputs (in `data/` folder)
* You generate the input data on you own laptop by running the script `generate_data_for_d2.py`
* `data/store_config.json`: A nested JSON file containing store metadata.
* `data/2026/...`: A folder hierarchy containing subdirectories with .log files in mixed character encodings.

Outputs (files your code should generate)
* `cleaned_transactions.csv`: A CSV table containing all parsed, outlier-cleaned, and currency-corrected transactions.
* `hourly_revenue.json`: A JSON report containing total revenue per hour per item.
* `status_pivot_table.csv`: A CSV file containing a pivot table of total transaction revenue grouped by item across status categories.
* `item_refund_ratios.csv`: A CSV table containing refund ratios for each item type.

## Task 1: File ingestion and regular expressions

1. Write a robust file reader function:
    * Use `pathlib.Path` to handle file paths.
    * The function should be able to handle both `UTF-8` and `ISO-8859-1/Latin-1` encoding. You may assume that these are the only two encodings the files can have. Note that the function should handle the encoding, you may not hard-code which encoding each file has.
    * Return a tuple containing two strings: `(file_text_content, encoding_used)`.
2. Write a log parser function that uses regular expressions.
    Log files contain valid sales lines mixed with system noise.<br>
    Valid line format:<br>
    `[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)`<br>
    Noise lines to ignore:<br>
    `--- PRINTER PAPER LOW WARNING ---`<br>
    `=== NETWORK SYNC OK ===`<br>
    * Construct a regular expression (`re` module) to capture the 6 key fields from valid lines:
        * `timestamp` (string, format: `YYYY-MM-DD HH:MM:SS`)
        * `register_id` (string, e.g., `POS_01`)
        * `item` (string, e.g., `Latte`)
        * `price` (float, e.g., `4.50`)
        * `currency` (string, e.g., `EUR`, `NOK`, `USD`)
        * `status` (string: `SUCCESS`, `REFUND`, or `CANCELLED`)
    * Return a list of Python dictionaries, where each dictionary represents one valid transaction record.

In [2]:
import json
import random
import re
from datetime import datetime, timedelta
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import TypedDict

import numpy as np
import pandas as pd
from numpy.typing import ArrayLike, NDArray


In [3]:
class Transaction(TypedDict):
    timestamp: str
    register_id: str
    item: str
    price: float
    currency: str
    status: str


SALES_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]"
    r"\s+\[(?P<register_id>[A-Za-z0-9_-]+)\]"
    r"\s+item=(?P<item>\S(?:.*?\S)?)"
    r"\s+price=(?P<price>[+-]?[0-9]+(?:\.[0-9]+)?)"
    r"\s+(?P<currency>[A-Z]{3})"
    r"\s+\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
)


def read_log_file(file_path: str | Path) -> tuple[str, str]:
    """Read a log file using UTF-8, falling back to Latin-1."""
    file_bytes = Path(file_path).read_bytes()

    try:
        return file_bytes.decode("utf-8-sig"), "utf-8"
    except UnicodeDecodeError:
        return file_bytes.decode("iso-8859-1"), "iso-8859-1"


def parse_log(file_text: str) -> list[Transaction]:
    """Extract valid transaction lines and ignore noise or malformed lines."""
    transactions: list[Transaction] = []

    for line in file_text.splitlines():
        match = SALES_PATTERN.fullmatch(line.strip())

        if match is None:
            continue

        transactions.append(
            Transaction(
                timestamp=match["timestamp"],
                register_id=match["register_id"],
                item=match["item"],
                price=float(match["price"]),
                currency=match["currency"],
                status=match["status"],
            )
        )

    return transactions

## Task 2: Recursive utility functions

1. Write a recursive directory search function that finds all log files in a nested directory and returns a list of `pathlib.Path` objects:
    * Convert the input path to a `pathlib.Path` object.
    * Iterate through items using `Path.iterdir()`.
    * If an item is a directory, recursively search inside it.
    * If an item is a file with a matching extension (`.log`), add its Path object to a list.
    * Constraint: Do NOT use `Path.glob()`, `Path.rglob()`, or `os.walk()`. You must implement explicit recursion.
2. Metadata about the different stores can be found in the `data/store_config.json` file. This file contains deeply nested dictionaries and lists (see example below). Write a recursive function that:
    * Traverses any combination of nested dictionaries and lists found in this file.
    * Returns a set (type hint: `set[str]`) of all register id strings where `"active": true`.

Example config.json:
```python
{
  "company": "Campus Coffee",
  "regions": [
    {
      "region_name": "Main Campus",
      "stores": [
        {
          "name": "Library Cafe",
          "registers": [
            {"id": "POS_01", "active": true},
            {"id": "POS_02", "active": false}
          ]
        }
      ]
    }
  ]
}
```

In [4]:
def find_log_files(directory: str | Path) -> list[Path]:
    """Recursively find .log files within a directory."""
    directory = Path(directory)
    log_files: list[Path] = []

    for item in sorted(directory.iterdir()):
        # Skip symbolic links to avoid recursive directory loops.
        """
        Jos hakemisto A sisältää symlinkin link_to_A, joka osoittaa takaisin hakemistoon A, 
        item.is_dir() palauttaa myös symlinkeille True. Tällöin funktio kutsuisi itseään loputtomiin 
        (find_log_files(link_to_A)), kunnes Python nostaa RecursionError-virheen.
        Hvis mappen A inneholder en symlink link_to_A som peker tilbake til mappen A, vil item.is_dir() også returnere 
        True for symlinker. I så fall vil funksjonen kalle seg selv i det uendelige (find_log_files(link_to_A)) 
        helt til Python utløser en RecursionError.
        When item.is_symlink() is True, continue tells Python to skip that symlink immediately 
        and jump straight to the next file or directory in sorted(directory.iterdir())
        """
        if item.is_symlink():
            continue
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix.lower() == ".log":
            log_files.append(item)

    return log_files


def find_active_registers(data: object) -> set[str]:
    """Collect register IDs marked active in nested dictionaries and lists."""
    active_registers: set[str] = set()

    if isinstance(data, dict):
        register_id = data.get("id")

        if data.get("active") is True and isinstance(register_id, str):
            active_registers.add(register_id)

        for value in data.values():
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers

## Task 3: Functions for cleaning and analysing data using NumPy
1. Write a function for removing price outliers that:
    * Converts input price data to a 1D NumPy float array.
    * Detects hardware glitch outliers ($999.00$) where a value deviates from the mean by more than 3 standard deviations:$$\lvert x - \mu \rvert > 3\sigma$$
    * Replaces detected outlier values with the median of the valid prices.
    * Returns the cleaned 1D array.
2. Write a function `compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray` that:
    * Computes the element-wise refund ratio using NumPy array operations:
    $$\text{Refund Ratio} = \frac{\text{Refund Count}}{\text{Sales Count} + 10^{-6}}$$
    * The addition of $\epsilon = 10^{-6}$ prevents zero-division errors.

In [5]:
def remove_price_outliers(prices: ArrayLike) -> NDArray[np.float64]:
    # Replace prices beyond three standard deviations with the valid median
    price_array = np.array(prices, dtype=np.float64).reshape(-1)

    if price_array.size == 0:
        return price_array

    if not np.all(np.isfinite(price_array)):
        raise ValueError("Prices must contain only finite numbers.")

    mean_price = np.mean(price_array)
    std_price = np.std(price_array)

    outlier_mask = np.abs(price_array - mean_price) > 3 * std_price

    if np.any(outlier_mask):
        valid_prices = price_array[~outlier_mask]
        price_array[outlier_mask] = np.median(valid_prices)

    return price_array


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    # Calculate element-wise refund ratios, adding epsilon to avoid division by zero
    sales = np.asarray(sales_count, dtype=np.float64)
    refunds = np.asarray(refund_count, dtype=np.float64)

    if sales.shape != refunds.shape:
        raise ValueError("Sales and refund counts must have the same shape.")

    epsilon = 1e-6
    return refunds / (sales + epsilon)

## Task 4: Data analysis using Pandas
1. Write a function that:
    * Converts the list of transaction dictionaries read from the log files into a `pandas.DataFrame`.
    * Converts the `timestamp` column to `datetime` and sets it as the DataFrame index.
    * Sorts the DataFrame chronologically by timestamp.
2. Write a function that has the DataFrame you just created as input, groups data by item and resamples transactions in 1-hour windows (1h), calculating total price (sum).
3. Write a function that has the DataFrame as input  and creates a pivot table displaying total transaction revenue grouped by item across status categories (`SUCCESS`, `REFUND`, `CANCELLED`).
4. Write a function that has the DataFrame as input and calculates the refund ratio per item by calling the function you made in task 3.

In [6]:
def create_transaction_dataframe(transactions: list[Transaction]) -> pd.DataFrame:
    """Create a chronologically sorted DataFrame with a datetime index."""
    columns = [
        "timestamp", "register_id", "item", "price", "currency", "status"
    ]
    dataframe = pd.DataFrame(transactions, columns=columns)

    dataframe["timestamp"] = pd.to_datetime(
        dataframe["timestamp"],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )
    dataframe["price"] = pd.to_numeric(
        dataframe["price"], errors="raise"
    )

    return dataframe.set_index("timestamp").sort_index()


def calculate_hourly_totals(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Sum transaction prices per item in one-hour windows."""
    # Ensure DatetimeIndex or pass 'on' parameter to resample
    resampled = dataframe.groupby("item").resample("1h")["price"]  # type: ignore

    return (
        resampled.sum()
        .rename("total_price")
        .reset_index()
    )


def create_revenue_pivot(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Sum transaction prices per item across all status categories."""
    statuses = ["SUCCESS", "REFUND", "CANCELLED"]

    return (
        dataframe.pivot_table(
            index="item",
            columns="status",
            values="price",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(columns=statuses, fill_value=0)
    )


def calculate_refund_ratio_per_item(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Calculate refund-to-success transaction count ratios per item."""
    counts = (
        dataframe.groupby(["item", "status"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["SUCCESS", "REFUND"], fill_value=0)
    )

    counts["refund_ratio"] = compute_refund_ratio(
        sales_count=counts["SUCCESS"].to_numpy(),
        refund_count=counts["REFUND"].to_numpy(),
    )

    return counts.rename(
        columns={
            "SUCCESS": "sales_count",
            "REFUND": "refund_count",
        }
    ).rename_axis(columns=None)

## Task 5: Create full workflow
Combine the functions you have made into a workflow that does the following:
1. Recursively find all log files and extract active register IDs.
2. Ingest text logs and parse records into a list of dictionaries.
3. Build the initial Pandas DataFrame.
4. Apply NumPy outlier cleaning to remove $999.00$ price glitches.
5. Apply currency correction: multiply price by $10.0$ for all transactions where `currency == "NOK"`.
6. Generate resampled reports, pivot tables and refund ratios.
7. Save all output files listed in the introduction (Scenario: Overview of inputs and outputs for this assignment).

In [15]:
PROJECT_DIR = Path.home() / "Documents" /  "Deliverable_2"
DATA_DIR = PROJECT_DIR / "data"
REPORT_DIR = PROJECT_DIR / "reports"

print(f"Project: {PROJECT_DIR}")
print(f"Inputs:  {DATA_DIR}")
print(f"Reports: {REPORT_DIR}")

def run_workflow(
    data_directory: str | Path,
    output_directory: str | Path
) -> dict[str, pd.DataFrame]:
    """Read, clean, analyse, and export active-register transactions."""
    data_directory = Path(data_directory)
    output_directory = Path(output_directory)

    # 1. Find log files and active register IDs.
    log_files = find_log_files(data_directory)

    config_text, _ = read_log_file(
        data_directory / "store_config.json"
    )
    active_registers = find_active_registers(
        json.loads(config_text)
    )

    # 2. Read and parse transactions.
    transactions: list[Transaction] = []

    for log_file in log_files:
        file_text, _ = read_log_file(log_file)
        transactions.extend(parse_log(file_text))

    # 3. Create the datetime-indexed DataFrame.
    dataframe = create_transaction_dataframe(transactions)

    dataframe = dataframe.loc[
        dataframe["register_id"].isin(active_registers)
    ].copy()

    if dataframe.empty:
        raise ValueError("No transactions found for active registers.")

    # 4. Clean prices separately for each item and currency.
    # Exclude known 999.00 glitches before statistical cleaning.
    price_groups = dataframe.groupby(["item", "currency"]).indices
    price_column = dataframe.columns.get_loc("price")

    for positions in price_groups.values():
        prices = dataframe.iloc[positions, price_column].to_numpy(
            dtype=float,
            copy=True
        )
        glitch_mask = prices == 999.00
        # Keep prices that are not the known 999.00 hardware glitch.
        valid_prices = prices[~glitch_mask]

        if valid_prices.size == 0:
            raise ValueError(
                "Cannot replace price glitches in a group "
                "containing no valid prices."
            )

        cleaned_prices = remove_price_outliers(valid_prices)

        prices[~glitch_mask] = cleaned_prices
        prices[glitch_mask] = np.median(cleaned_prices)

        # Positional assignment handles repeated timestamps safely.
        dataframe.iloc[positions, price_column] = prices

    # 5. Correct NOK prices after cleaning.
    nok_mask = dataframe["currency"] == "NOK"
    dataframe.loc[nok_mask, "price"] *= 10.0

    # 6. Generate hourly totals and status pivots by currency.
    hourly_reports: list[pd.DataFrame] = []
    pivot_reports: list[pd.DataFrame] = []

    for currency, currency_data in dataframe.groupby("currency"):
        hourly_report = calculate_hourly_totals(currency_data)
        hourly_report = hourly_report.rename(
            columns={"total_price": "total_revenue"}
        )
        hourly_report["currency"] = currency
        hourly_reports.append(hourly_report)

        pivot_report = create_revenue_pivot(
            currency_data
        ).reset_index()

        pivot_report["currency"] = currency
        pivot_report.columns.name = None
        pivot_reports.append(pivot_report)

    hourly_revenue = (
        pd.concat(hourly_reports, ignore_index=True)
        .sort_values(["timestamp", "item", "currency"])
        .reset_index(drop=True)
    )

    status_pivot_table = (
        pd.concat(pivot_reports, ignore_index=True)
        .sort_values(["item", "currency"])
        .reset_index(drop=True)
    )

    # This Task 4 function calls compute_refund_ratio() from Task 3.
    item_refund_ratios = calculate_refund_ratio_per_item(dataframe)

    # 7. Save exactly the four required output files.
    output_directory.mkdir(parents=True, exist_ok=True)

    dataframe.to_csv(
        output_directory / "cleaned_transactions.csv",
        index=True,
        encoding="utf-8"
    )

    hourly_revenue.to_json(
        output_directory / "hourly_revenue.json",
        orient="records",
        date_format="iso",
        indent=2
    )

    status_pivot_table.to_csv(
        output_directory / "status_pivot_table.csv",
        index=False,
        encoding="utf-8"
    )

    item_refund_ratios.to_csv(
        output_directory / "item_refund_ratios.csv",
        index=True,
        encoding="utf-8"
    )

    return {
        "cleaned_transactions": dataframe,
        "hourly_revenue": hourly_revenue,
        "status_pivot_table": status_pivot_table,
        "item_refund_ratios": item_refund_ratios
    }

Project: /home/tyyniuni/Documents/Deliverable_2
Inputs:  /home/tyyniuni/Documents/Deliverable_2/data
Reports: /home/tyyniuni/Documents/Deliverable_2/reports


In [16]:
reports = run_workflow(DATA_DIR, REPORT_DIR)
cleaned = reports["cleaned_transactions"]

assert len(find_log_files(DATA_DIR)) == 9
assert len(cleaned) == 720
assert cleaned.index.is_monotonic_increasing
assert set(cleaned["currency"]) == {"EUR", "NOK", "USD"}
assert not cleaned["price"].isin([999.0, 9990.0]).any()
assert cleaned.loc[cleaned["currency"] == "NOK", "price"].between(25, 80).all()
assert cleaned.loc[cleaned["currency"] != "NOK", "price"].between(2.5, 8).all()

expected_files = {
    "cleaned_transactions.csv", "hourly_revenue.json",
    "status_pivot_table.csv", "item_refund_ratios.csv"
}
assert expected_files.issubset({p.name for p in REPORT_DIR.iterdir()})

# Totals must agree between the cleaned data, hourly report, and pivot table.
expected_totals = cleaned.groupby("currency")["price"].sum().sort_index()
hourly_totals = (
    reports["hourly_revenue"].groupby("currency")["total_revenue"].sum().sort_index()
)
pivot_totals = (
    reports["status_pivot_table"].set_index("currency")
    [["SUCCESS", "REFUND", "CANCELLED"]].sum(axis=1)
    .groupby(level="currency").sum().sort_index()
)
np.testing.assert_allclose(hourly_totals, expected_totals)
np.testing.assert_allclose(pivot_totals, expected_totals)

ratios = reports["item_refund_ratios"]
assert ratios["sales_count"].sum() == (cleaned["status"] == "SUCCESS").sum()
assert ratios["refund_count"].sum() == (cleaned["status"] == "REFUND").sum()
assert len(pd.read_csv(REPORT_DIR / "cleaned_transactions.csv")) == 720

print("Deployment test passed: 9 log files, 720 cleaned transactions.")
print("Hourly totals and status totals match for every currency.")
print("\nRequired output files:")
for filename in sorted(expected_files):
    print(f"  {filename}")
print("\nFirst five cleaned transactions:")
print(cleaned.head().to_string())
print("\nRefund ratios per item:")
print(ratios.to_string())


FileNotFoundError: [Errno 2] No such file or directory: '/home/tyyniuni/Documents/Deliverable_2/data'

# Path Setting, Deployment and Testing

In [8]:
"""
Dataset Generator for Campus Coffee Shop Assignment.
Creates nested folders, mixed encoding logs, and nested config JSON.
"""

import json
import random
from datetime import datetime, timedelta
from pathlib import Path

BASE_DIR = Path("C:\\", "Users", "tyyniuni", "Documents", "folder_for_saving", "d2_data")  # Path("data")
STORES = ["library_cafe", "science_building", "student_union"]
REGISTERS = ["POS_01", "POS_02", "POS_03"]
CURRENCIES = ["EUR", "NOK", "USD"]
ITEMS = [
    ("Espresso", 3.00),
    ("Latte", 4.50),
    ("Cappuccino", 4.20),
    ("Muffin", 3.50),
    ("Croissant", 3.80),
    ("Sandwich", 7.50),
]
STATUSES = ["SUCCESS", "SUCCESS", "SUCCESS", "SUCCESS", "REFUND", "CANCELLED"]

NOISE_LINES = [
    "--- PRINTER PAPER LOW ---",
    "[DEBUG] BARCODE SCANNER REBOOTED",
    "=== NETWORK SYNC OK ===",
]


def generate_log_line(ts: datetime, reg_id: str) -> str:
    item_name, base_price = random.choice(ITEMS)
    currency = random.choice(CURRENCIES)

    price = 999.00 if random.random() < 0.03 else round(base_price + random.uniform(-0.5, 0.5), 2)
    status = random.choice(STATUSES)
    ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")

    return f"[{ts_str}] [{reg_id}] item={item_name} price={price:.2f} {currency} ({status})\n"


def create_dataset():
    start_time = datetime(2026, 10, 15, 7, 0, 0)
    file_idx = 0

    for store in STORES:
        for reg in REGISTERS:
            folder = BASE_DIR / "2026" / store / reg
            folder.mkdir(parents=True, exist_ok=True)

            log_file = folder / "register_sales.log"
            encoding = "utf-8" if file_idx % 2 == 0 else "iso-8859-1"
            
            lines = []
            current_time = start_time

            for _ in range(80):
                current_time += timedelta(seconds=random.randint(10, 180))
                if random.random() < 0.15:
                    lines.append(random.choice(NOISE_LINES) + "\n")
                lines.append(generate_log_line(current_time, reg))

            with open(log_file, "w", encoding=encoding) as f:
                f.writelines(lines)

            file_idx += 1
            print(f"Generated: {log_file} [{encoding.upper()}]")

    # Generate nested JSON config
    config = {
        "company": "Campus Coffee",
        "regions": [
            {
                "region_name": "Main Campus",
                "stores": [
                    {
                        "name": "Library Cafe",
                        "registers": [
                            {"id": "POS_01", "active": True},
                            {"id": "POS_02", "active": True},
                        ],
                    },
                    {
                        "name": "Science Building",
                        "registers": [
                            {"id": "POS_03", "active": True},
                            {"id": "POS_04", "active": False},
                        ],
                    },
                ],
            }
        ],
    }

    with open(BASE_DIR / "store_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    print("\nDataset generation complete!")


if __name__ == "__main__":
    create_dataset()


Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/library_cafe/POS_01/register_sales.log [UTF-8]
Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/library_cafe/POS_02/register_sales.log [ISO-8859-1]
Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/library_cafe/POS_03/register_sales.log [UTF-8]
Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/science_building/POS_01/register_sales.log [ISO-8859-1]
Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/science_building/POS_02/register_sales.log [UTF-8]
Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/science_building/POS_03/register_sales.log [ISO-8859-1]
Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/student_union/POS_01/register_sales.log [UTF-8]
Generated: C:\/Users/tyyniuni/Documents/folder_for_saving/d2_data/2026/student_union/POS_02/register_sales.log [ISO-8859-1]
Generated: C:\/Users/t